# 资源分配问题

In [1]:
import numpy as np
import copy
import random
from time import time


## 类定义
定义两个类，node表示节点类， net表示线网类

In [2]:
class node():
    def __init__(self, ind=None, name=None, ziyuan=None, assignment=None):
        self.id = ind
        self.name = name
        self.ziyuan = ziyuan         # 节点所用资源的列表(在当前题目要求下，资源总和即可，不需要列表)
        self.assignment = assignment  # 节点被分配到的FPGA序号
        self.net = []               # 节点所属线网的序号列表


class net():
    def __init__(self, nodelist=None):
        self.nodelist = nodelist        # 线网中的节点的列表，此列表中每个元素是节点的名字
        self.assignment = set()    # 线网中的节点被分配到的FPGA的集合， 此集合中每个元素是FPGA的序号


## 定义函数
### 文件信息读取
读取文件，取得节点资源信息

In [3]:
def readNodeFile(filename):
    f = open(filename)
    vertex = {}        # 字典
    vertexname=[]      # 列表
    ind = 0
    for line in f:
        list1 = line.split()
        nm = list1[0]
        vertexname.append(nm)
        ziyuan = list(map(int, list1[-10:]))
        vertex[nm] = node(ind, nm, sum(ziyuan))
        ind += 1
    f.close()
    return vertex,vertexname

读取文件，取得线网信息

In [4]:

def readNetFile(filename,vertex):
    f = open(filename)
    linestack=[]
    for line in f:
        linestack.append(line.split())
    nets=[]
    nodelist=[]
    ind=0
    while linestack!=[]:
        line = linestack.pop()
        nodelist.append(line[0])
        if 's' in line:
            nets.append(net(nodelist))
            for node in nodelist:
                vertex[node].net.append(ind)
            nodelist=[]
            ind += 1
    f.close()
    return nets

### 定义分配函数fenpeiMethod
这里仅仅用于demo，采用随机分配

In [5]:
def fenpeiMethod(vertex, nosFPGA,vertexname):
    return randomFenpei(vertex, nosFPGA,vertexname)

def randomFenpei(vertex, nosFPGA,vertexname):
    nosnode = len(vertex)
    fenpei=[]
    for i in range(nosnode):
        fenpei.append(np.random.randint(nosFPGA))
    F = [[] for i in range(nosFPGA)]
    for i in range(nosnode):
        F[fenpei[i]].append(i)
        vertex[vertexname[i]].assignment = fenpei[i]
        for onenet in nets:
            if vertexname[i] in onenet.nodelist:
                onenet.assignment.add(fenpei[i])    
    return F

### 定义写文件函数writeResult

In [6]:
def writeResult(F,vertexname):
    nosFPGA = len(F)
    for fpga in range(nosFPGA):
        f.write("F"+str(fpga))
        for onenode in F[fpga]:
            f.write(" "+vertexname[onenode])
        f.write("\n")

### 定义资源差异函数 ziyuanCal
计算各个板的资源

In [7]:
def ziyuanCal(F,vertex,vertexname):
    nosFPGA = len(F)
    Z = []
    for i in range(nosFPGA):
        list1 = F[i]
        print(list1)
        sumzy = 0
        for ind in list1:
            sumzy += vertex[vertexname[ind]].ziyuan
        Z.append(sumzy)
    return Z


### 定义板间连线计算函数 linkCal
计算FPGA板间连线的总和

In [8]:
def linkCal(nets,nosFPGA):
    L = [0 for i in range(nosFPGA)]
    for onenet in nets:
        if len(onenet.assignment)==1:
            continue
        for fpgaInd in onenet.assignment:
            L[fpgaInd] += 1
    return L


### 清空分配信息
包括节点和线网中的分配信息

In [9]:
def clearAssignInfo(vertex,vertexname,nets):
    for nodeInd in range(len(vertexname)):
        vertex[vertexname[nodeInd]].assignment = None
    for onenet in nets:
        onenet.assignment = set()

## 执行代码
读取文件

In [10]:
# 读取节点文件
filename = "./contest/testdata-0/design.are"
vertex,vertexname = readNodeFile(filename)

In [11]:
# 读取线网文件
filename = "./contest/testdata-0/design.net"
nets = readNetFile(filename,vertex)

In [12]:
print(nets[3].nodelist)
print(nets[4].assignment)

['g25', 'g24', 'g23', 'g26']
set()


设定分块数量，即FPGA板的数量

In [13]:
nosFPGA = 4 

把结果文件打开

In [14]:
filename = "./result.res"
f = open(filename,'w')
# np.random.seed(np.random.randint(100))

超参数设置：

In [15]:
# showPlot = 1  # showPlot=1，显示最后的最佳路径图片
N = 10		# 染色体群体规模
MAXITER = 10**3  # 最大循环次数
SelfAdj = 0  # SelfAdj = 1时为自适应
# randAccordDist = 0  # 在mutate的时候根据位置值分配对应的概率，此概率决定哪个城市参与mutate

pc = 0.15		# 交叉概率
pw = 0.85		# 变异概率


第1种情况：

In [16]:
clearAssignInfo(vertex,vertexname,nets)
F = fenpeiMethod(vertex,nosFPGA,vertexname)
writeResult(F,vertexname)
f.write(str(np.std(ziyuanCal(F,vertex,vertexname)))+"\n")

[2, 4, 5, 6, 8, 11, 12, 13, 14, 16, 21, 26, 32, 33, 35, 40, 41, 49, 50]
[15, 23, 24, 28, 29, 30, 31]
[0, 10, 17, 18, 20, 25, 34, 36, 37, 38, 43, 44, 51, 52]
[1, 3, 7, 9, 19, 22, 27, 39, 42, 45, 46, 47, 48]


17

第2种情况：

In [17]:
#################################
# 产生N个染色体的初始群体,保存在pop里
# 每个染色体代表TSP问题的某一个解（即所有城市都经过一次的轨迹）

def initPop(N, vertex, nosFPGA):
    pop = np.zeros([N, len(vertex)])
    for i in range(N):  # 使用随机函数生成N个染色体
        lst = list(range(1, len(vertex),))
        random.shuffle(lst)
        pop[i, range(1, len(vertex))] = lst
    pop[range(N), 0] = 0  # 所有染色体都从城市0开始，最后回到城市0.
    return pop


#################################
# 根据染色体群体pop已经对应的适应值fitness，
# 找到最高的适应值f，以及对应的染色体bst和其在pop中的编号/下标ind

def findBest(pop, fitness):
    f = np.max(fitness)
    ind = np.asarray(np.where(fitness == f)).flatten()
    bst = pop[ind, :]
    return [bst, f, ind]


#################################
# 根据染色体的适应值，按照一定的概率，生成新一代染色体群体newpop

def chooseNewP(pop, fitness):
    N, nc = np.shape(pop)
    fitness = np.cumsum(fitness)
    lst = np.zeros([N, 1])
    rvalLst = np.random.rand(N, 1)
    for i in range(N):
        rval = np.random.rand()
        lst[i] = 0
        for j in range(N-1, -1, -1):
            if rval > fitness[j]:
                lst[i] = j
                break
    newpop = pop[list(lst.flatten().astype(np.uint8)), :]
    return newpop


#################################
# 根据交叉概率pc，以及各染色体的适应值fitness，通过交叉的方式生成新群体
# #SelfAdj = 1时为自适应，否则取固定的交叉概率pc

def crossPop(pop, pc, fitness, SelfAdj):
    N, nc = np.shape(pop)
    lst = list(range(N))
    np.random.shuffle(lst)

    fm = np.max(fitness)
    fa = np.mean(fitness)
    k1 = pc
    k2 = pc
    i = 0
    while i < int(N/2):
        rval = np.random.rand()
        j = np.random.randint(int(N/2), N)
        if SelfAdj == 1: 		# 自适应，参考149页
            fprime = np.max(fitness[i], fitness[j])
            if fprime > fa:
                pc = k1*(fm-fprime) / (fm-fa)
            else:
                pc = k2
        if rval > pc:
            continue
        partner1 = copy.copy(pop[lst[i], :])
        partner2 = copy.copy(pop[lst[j], :])
        if (partner1 == partner2).all():
            continue
        child1, child2 = genecross(partner1, partner2)
        pop[lst[i], :] = copy.copy(child1)
        pop[lst[j], :] = copy.copy(child2)
        i = i+1


#################################
# 父染色体partner1,partner2，通过交叉方式
# 生成两个子染色体child1,child2

def genecross(partner1, partner2):
    length = len(partner1)
    idx1 = 0
    idx2 = 0
    while idx1 == idx2:
        idx1 = random.randint(0, length-1)
        idx2 = random.randint(0, length-1)
    ind1 = min(idx1, idx2)
    ind2 = max(idx1, idx2)

    child1 = copy.copy(partner1)
    child2 = copy.copy(partner2)

    tem1 = copy.copy(child1[ind1:ind2])
    tem2 = copy.copy(child2[ind1:ind2])

    if (tem1 == tem2).all():
        return [child1, child2]
    if set(tem1) == set(tem2):
        child1[ind1:ind2] = tem2
        child2[ind1:ind2] = tem1
        return [child1, child2]

    temdff1 = set(tem1).difference(set(tem2))
    temdff2 = set(tem2).difference(set(tem1))

    for i in range(len(temdff1)):
        child1[np.where(child1 == list(temdff2)[i])] = list(temdff1)[i]
        child2[np.where(child2 == list(temdff1)[i])] = list(temdff2)[i]

    child1[ind1:ind2] = tem2
    child2[ind1:ind2] = tem1
    return [child1, child2]


#################################
# 根据变异概率pw，以及各染色体的适应值fitness，通过变异的方式生成新群体
# #SelfAdj = 1时为自适应，否则取固定的变异概率pw

def mutPop(pop, pw, fitness, SelfAdj):
    N, nc = np.shape(pop)

    fm = max(fitness)
    fa = np.mean(fitness)
    k3 = pw
    k4 = pw
    for i in range(N):
        rval = random.random()
        if SelfAdj == 1: 		# 自适应，参考149页
            if fitness[i] > fa:
                pw = k3*(fm-fitness[i]) / (fm-fa)
            else:
                pw = k4
        if rval > pw:
            continue
        pop[i, :] = np.asarray(mutDistGene(pop[i, :]))


# 在gene的nc个城市中，根据距离值找出最不合理的城市valm以及其下标indx
# 把该城市valm从原有轨迹中抽取出来（其两边的城市直接相连），并把valm城市
# insert到列表的indy的位置。

def mutDistGene(gene):
    global R
    global randAccordDist
    nc = len(gene)
    distGene = [R[gene[np.mod(i, nc)].astype(np.uint8), gene[np.mod(i+1, nc)].astype(np.uint8)]
                for i in range(nc)]  # gene中相邻城市之间的距离,gene[0]和gene[1]的距离保存在distGene[0]中
    distGeneR = [R[gene[np.mod(i, nc)].astype(np.uint8), gene[np.mod(i-1, nc)].astype(np.uint8)]
                 for i in range(nc)]  # gene中相邻城市之间的距离,gene[0]和gene[nc-1]的距离保存在distGeneR[0]中
    distGene = list(np.asarray(distGene) + np.asarray(distGeneR))
    divDist = [R[gene[np.mod(i, nc)].astype(np.uint8), :].sum()
               for i in range(nc)]
    distPercGene = [distGene[i]/divDist[i] for i in range(nc)]

    indx = np.argmax(distPercGene)
    listgene = list(gene)
    valm = int(listgene.pop(indx))
    alist = range(nc)
    clist = [str(alist[i]) for i in range(nc)]
    rList = copy.copy(R[valm, :])
    maxrList = max(rList)
    rList[valm] = 10**10  # 原数值为0，换成一个非常大（足够大）的数值
    try:
        rList = (np.exp(maxrList/rList)).astype(np.uint64)
    except:
        print('df')
    rList[valm] = 0  # 以便使得valm不会被重复选择

    ind1 = ind2 = valm
    while ind2 == valm:
        if randAccordDist == 1:
            dumm1, dumm2, ind1, ind2 = findTwoMaxRandom(list(rList))
        else:
            dumm1, dumm2, ind1, ind2 = findTwoMax(list(rList))

    inda = listgene.index(ind1)
    indb = listgene.index(ind2)
    if np.mod(inda-indb, nc) <= np.mod(indb-inda, nc):
        indy = np.mod(inda, nc)
    else:
        indy = np.mod(inda+1, nc)

    listgene.insert(indy, valm)
    if listgene[0] != 0:
        cycleList(listgene)
    return listgene

 # 旋转队列lst，使得数值0在lst[0]的位置


def cycleList(lst):
    temlst = []
    while lst[0] != 0:
        temlst.append(lst.pop(0))
    lst.extend(temlst)


def findTwoMax(lst):
    ind1 = np.argmax(lst)
    max1 = lst[ind1]

    lst[ind1] = min(lst)

    ind2 = np.argmax(lst)
    max2 = lst[ind2]
    return [max1, max2, ind1, ind2]


# 按照一定的规则，使得lst的最大和次最大的两个数值及其索引被输出的概率最大
# lst每个元素必须是正数,表示权值

def findTwoMaxRandom(lst):
    indxList = [str(i) for i in range(len(lst))]
    ind1 = np.argmax(lst)
    max1 = lst[ind1]
    lst[ind1] = min(lst)
    ind2 = int(weight_choice(indxList, lst))		# 概率
    max2 = lst[ind2]
    return [max1, max2, ind1, ind2]


# lst: 待选取序列
# weight: lst对应的权重序列

def weight_choice(lst, weight):
    new_lst = []
    for i, val in enumerate(lst):
        new_lst.extend(val * weight[i])
    return random.choice(new_lst)


# 步骤1，产生N个染色体的初始群体,保存在pop里面
pop = initPop(N, vertex, nosFPGA)  # pop 遗传算法中的种群

iter = 0
tstart = time()
while iter < MAXITER:
    iter = iter+1

    LinkLength = linkCal(pop, nosFPGA)  # 步骤2：计算每个染色体的板间连线长度

    fitness = 1.0/LinkLength  # 计算每个染色体的适应值
    b, f, ind = findBest(pop, fitness)			# 找到在当前种群中，适应度最高的个体

    # # 步骤3：如果满足某些标准，算法停止
    # if 0:
    #     break
    chooseNewP(pop, fitness)  # 否则，选取出一个新的群体
    crossPop(pop, pc, fitness, SelfAdj)  # 步骤4：交叉产生新染色体，得到新群体
    mutPop(pop, pw, fitness, SelfAdj)  # 步骤5：基因变异
    pop[0, :] = b[0, :]						# 保留上一代中适应值最高的染色体

    if np.mod(iter, MAXITER/10) == 1:
        titer = time()
        print("iter = %d after running %d seconds with route-length=%f" %
              (iter, int(titer-tstart), LinkLength))

clearAssignInfo(vertex, vertexname, nets)
F = fenpeiMethod(vertex, nosFPGA, vertexname)
writeResult(F, vertexname)
f.write(str(sum(linkCal(nets, nosFPGA)))+"\n")


3

第3种情况：

In [18]:
clearAssignInfo(vertex,vertexname,nets)
F = fenpeiMethod(vertex,nosFPGA,vertexname)
writeResult(F,vertexname)
f.write(str(sum(linkCal(nets,nosFPGA)))+"\n")

3

关闭文件

In [19]:
f.close()

## 各种测试语句：

In [20]:
print(linkCal(nets,nosFPGA))

[22, 29, 18, 29]


In [21]:
print(F)

[[0, 5, 6, 13, 18, 21, 27, 29, 37, 42, 46], [3, 4, 7, 8, 12, 15, 20, 22, 25, 33, 35, 36, 38, 43, 44, 49, 51], [2, 14, 16, 19, 28, 31, 34, 39, 48, 52], [1, 9, 10, 11, 17, 23, 24, 26, 30, 32, 40, 41, 45, 47, 50]]


In [22]:
print(ziyuanCal(F,vertex,vertexname))

[0, 5, 6, 13, 18, 21, 27, 29, 37, 42, 46]
[3, 4, 7, 8, 12, 15, 20, 22, 25, 33, 35, 36, 38, 43, 44, 49, 51]
[2, 14, 16, 19, 28, 31, 34, 39, 48, 52]
[1, 9, 10, 11, 17, 23, 24, 26, 30, 32, 40, 41, 45, 47, 50]
[158, 151, 90, 98]


In [23]:
np.std([276, 100, 24, 97])

92.74797841462637

In [24]:
f=[2, 7, 24, 26, 36, 43, 45, 49, 52]
[vertex[vertexname[ind]].ziyuan for ind in f]

[65, 6, 6, 3, 1, 1, 1, 1, 1]

In [25]:
sum([65, 6, 6, 3, 1, 1, 1, 1, 1])

85

In [26]:
f=[5, 6, 10, 14, 15, 16, 19, 27, 30, 31, 32, 40, 41, 44, 46]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

110

In [27]:
f=[0, 1, 9, 11, 13, 17, 18, 25, 28, 29, 33, 34, 35, 37, 38, 42]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

162

In [28]:
f=[3, 4, 8, 12, 20, 21, 22, 23, 39, 47, 48, 50, 51]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

140

In [29]:
np.std([85,110,162,140])

29.22648627529488